In [0]:
# ============================================================
# FIRMABLE - TEAM-AWARE BRONZE VALIDATION
# ============================================================

from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.firmable_v2_bronze"

companies = spark.table(f"{BRONZE}.companies")
employees = spark.table(f"{BRONZE}.employees_raw")
ingestion = spark.table(f"{BRONZE}.employees_ingestion_raw")
ground_truth = spark.table(f"{BRONZE}.ground_truth")

results = []

def check(name, condition, details):
    results.append((
        name,
        "PASS" if condition else "FAIL",
        str(details)
    ))

# 1. Company and employee integrity
company_count = companies.count()
employee_count = employees.count()

check("500 companies", company_count == 500, company_count)

check(
    "Unique employee IDs",
    employee_count == employees.select("employee_id").distinct().count(),
    employee_count
)

check(
    "No duplicate employee records",
    employees.groupBy("company_id", "employee_id")
    .count().filter("count > 1").count() == 0,
    "Duplicate employee keys checked"
)

# 2. Company employee-count reconciliation
company_counts = (
    employees.groupBy("company_id")
    .count()
    .withColumnRenamed("count", "actual_count")
)

mismatches = (
    companies.join(company_counts, "company_id", "left")
    .filter(
        F.col("actual_count") != F.col("employee_count")
    )
    .count()
)

check(
    "Company employee counts match",
    mismatches == 0,
    f"{mismatches} mismatched companies"
)

# 3. Team integrity
team_employees = employees.filter(
    F.col("team_id").isNotNull()
)

team_count = team_employees.select(
    "company_id", "team_id"
).distinct().count()

team_name_errors = (
    team_employees
    .filter(
        F.col("team_name").isNull() |
        F.col("department").isNull()
    )
    .count()
)

team_department_errors = (
    team_employees
    .groupBy("company_id", "team_id")
    .agg(
        F.countDistinct("department").alias("department_count"),
        F.countDistinct("team_name").alias("team_name_count")
    )
    .filter(
        (F.col("department_count") != 1) |
        (F.col("team_name_count") != 1)
    )
    .count()
)

check(
    "Teams have valid names and departments",
    team_name_errors == 0,
    team_name_errors
)

check(
    "Each team belongs to one department",
    team_department_errors == 0,
    team_department_errors
)

check(
    "Teams generated",
    team_count > 0,
    f"{team_count} unique teams"
)

# 4. Team manager and lead representation
team_role_summary = (
    team_employees
    .groupBy("company_id", "team_id")
    .agg(
        F.sum(
            F.when(
                F.col("job_title").rlike("(?i).*Manager$"),
                1
            ).otherwise(0)
        ).alias("manager_count"),
        F.sum(
            F.when(
                F.col("job_title").rlike(
                    "(?i).*Lead.*|.*Lead$"
                ),
                1
            ).otherwise(0)
        ).alias("lead_count")
    )
)

# Flat companies have team leads but no team managers.
flat_company_ids = (
    companies.filter(F.col("org_structure") == "Flat")
    .select("company_id")
)

flat_teams = team_role_summary.join(
    flat_company_ids, "company_id"
)

non_flat_teams = team_role_summary.join(
    flat_company_ids, "company_id", "left_anti"
)

flat_role_errors = flat_teams.filter(
    F.col("lead_count") < 1
).count()

non_flat_role_errors = non_flat_teams.filter(
    (F.col("manager_count") < 1) |
    (F.col("lead_count") < 1)
).count()

check(
    "Flat teams contain team leads",
    flat_role_errors == 0,
    flat_role_errors
)

check(
    "Non-flat teams contain managers and leads",
    non_flat_role_errors == 0,
    non_flat_role_errors
)

# 5. Ground truth integrity
gt_count = ground_truth.count()

root_count = ground_truth.filter(
    F.col("actual_manager_id").isNull()
).count()

duplicate_gt = (
    ground_truth.groupBy("company_id", "employee_id")
    .count().filter("count > 1").count()
)

missing_gt = (
    employees.select("company_id", "employee_id")
    .join(
        ground_truth.select("company_id", "employee_id"),
        ["company_id", "employee_id"],
        "left_anti"
    )
    .count()
)

invalid_manager = (
    ground_truth.alias("g")
    .filter(F.col("actual_manager_id").isNotNull())
    .join(
        employees.alias("m"),
        (
            (F.col("g.company_id") == F.col("m.company_id")) &
            (F.col("g.actual_manager_id") == F.col("m.employee_id"))
        ),
        "left_anti"
    )
    .count()
)

check(
    "Ground truth covers all employees",
    gt_count == employee_count and missing_gt == 0,
    f"{gt_count} records; {missing_gt} missing"
)

check(
    "Exactly one root per company",
    root_count == company_count,
    root_count
)

check(
    "No duplicate ground truth records",
    duplicate_gt == 0,
    duplicate_gt
)

check(
    "All manager references exist",
    invalid_manager == 0,
    invalid_manager
)

# 6. Messy ingestion integrity
ingestion_count = ingestion.count()

check(
    "Ingestion count matches employee count",
    ingestion_count == employee_count,
    ingestion_count
)

check(
    "Team information preserved in ingestion",
    {"team_id", "team_name"}.issubset(set(ingestion.columns)),
    ingestion.columns
)

# 7. Ensure inference data has no actual manager ID
forbidden = {
    "actual_manager_id",
    "manager_id",
    "reporting_to"
}

exposed = (
    set(employees.columns) | set(ingestion.columns)
) & forbidden

check(
    "Ground truth isolated from inference",
    len(exposed) == 0,
    exposed
)

# 8. Report
validation_df = spark.createDataFrame(
    results,
    ["validation_check", "status", "details"]
)

print("TEAM-AWARE BRONZE VALIDATION")
print("============================")

display(validation_df)

print("\nDATASET SUMMARY")
print("---------------")
print(f"Companies: {company_count:,}")
print(f"Employees: {employee_count:,}")
print(f"Ground truth records: {gt_count:,}")
print(f"Actual reporting relationships: {gt_count-root_count:,}")
print(f"Unique teams: {team_count:,}")
print(f"Employees with team IDs: {team_employees.count():,}")
print(f"Employees without team IDs: {employee_count-team_employees.count():,}")

failed = [r for r in results if r[1] == "FAIL"]

print("\nRESULT:", "FAILED CHECKS FOUND" if failed else "ALL CHECKS PASSED")

for item in failed:
    print(item)